In [1]:
import pandas as pd
from nem_battery.features import split_by_time
from nem_battery.load import INTERIM_DIR, PROJECT_ROOT
from nem_battery.features import to_half_hourly, add_features, FEATURES, TARGET

df = pd.read_parquet(INTERIM_DIR / "price_demand_sa1.parquet")
hh = to_half_hourly(df)

# round 5min to 30min
print("Rows:", len(hh))
print(hh.groupby(hh["interval_start"].dt.date).size().value_counts())
hh.head()

Rows: 87648
48    1826
Name: count, dtype: int64


,interval_start,rrp,total_demand_mw
0,2021-10-01 00:00:00,9.813333,1336.116667
1,2021-10-01 00:30:00,2.773333,1319.851667
2,2021-10-01 01:00:00,5.590000,1286.221667
3,2021-10-01 01:30:00,5.293333,1227.813333
4,2021-10-01 02:00:00,-0.756667,1181.835000


In [2]:
# build features
feat = add_features(hh)
print(feat.shape)
feat[["interval_start", "rrp"] + FEATURES].iloc[340:345]

(87648, 15)


,interval_start,rrp,slot,dayofweek,month,is_weekend,rrp_lag_1d,rrp_lag_7d,demand_lag_1d,rrp_slot_median_7d,prev_day_mean,prev_day_max,prev_day_neg_share
340,2021-10-08 02:00:00,56.015000,4,4,10,0,-5.250000,-0.756667,1059.540000,-0.756667,20.232951,153.018333,0.5
341,2021-10-08 02:30:00,47.256667,5,4,10,0,-0.518333,-1.811667,1029.486667,-0.518333,20.232951,153.018333,0.5
342,2021-10-08 03:00:00,44.925000,6,4,10,0,-2.343333,-0.363333,990.015000,-0.363333,20.232951,153.018333,0.5
343,2021-10-08 03:30:00,44.295000,7,4,10,0,-0.310000,0.686667,1006.473333,0.686667,20.232951,153.018333,0.5
344,2021-10-08 04:00:00,51.688333,8,4,10,0,5.246667,0.318333,1008.980000,5.246667,20.232951,153.018333,0.5


In [3]:
# leakage check
target_time = pd.Timestamp("2025-02-12 18:30")
row = feat.loc[feat["interval_start"] == target_time, FEATURES + [TARGET]]
print(row.T)

# Manual checks against the raw half-hourly data
yesterday = target_time - pd.Timedelta(days=1)
last_week = target_time - pd.Timedelta(days=7)
prev_day = hh[hh["interval_start"].dt.normalize() == pd.Timestamp("2025-02-11")]

print("\nExpected rrp_lag_1d:   ", hh.loc[hh["interval_start"] == yesterday, "rrp"].values)
print("Expected rrp_lag_7d:   ", hh.loc[hh["interval_start"] == last_week, "rrp"].values)
print("Expected prev_day_max: ", prev_day["rrp"].max())

                           59077
slot                   37.000000
dayofweek               2.000000
month                   2.000000
is_weekend              0.000000
rrp_lag_1d            267.316667
rrp_lag_7d            150.831667
demand_lag_1d        2957.720000
rrp_slot_median_7d    150.831667
prev_day_mean         141.348090
prev_day_max         1897.553333
prev_day_neg_share      0.000000
rrp                 12722.853333

Expected rrp_lag_1d:    [267.31666667]
Expected rrp_lag_7d:    [150.83166667]
Expected prev_day_max:  1897.5533333333333


In [4]:
# drop warmup rows
model_df = feat.dropna(subset=FEATURES).reset_index(drop=True)

out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
model_df.to_parquet(out_dir / "features_sa1_30min.parquet", index=False)

In [ ]:
# Train model and evaluate


In [5]:
train, valid, test = split_by_time(model_df)

for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5}: {len(part):6,} rows  {part['date'].min().date()} → {part['date'].max().date()}")

# Sanity checks
assert len(train) + len(valid) + len(test) == len(model_df)   # nothing lost
assert train["date"].max() < valid["date"].min()                 # no overlap
assert valid["date"].max() < test["date"].min()

train: 52,272 rows  2021-10-08 → 2024-09-30
valid: 17,520 rows  2024-10-01 → 2025-09-30
test : 17,520 rows  2025-10-01 → 2026-09-30
